# Compare data before and after a filter

Compares a **before** dataset with an **after** dataset (the same data with a filter applied, for example
removing a job role's values), matched on key columns. Four views: rows the filter nulled, the trendline
over time, the value distribution, and the effect on the final estimates column. Each is also split by
whether the row belongs to the filtered values (`FILTERED_VALUES` of `FILTER_COLUMN`), so you can see the
filter's intended effect and any spill-over onto other rows.

**Before running:** both datasets exist in S3 and the parameters below are set. Data is read lazily and
only small aggregates are collected. This notebook only reads from S3; it writes nothing back.

**Stop the notebook instance when you finish** — SageMaker bills while it is running.

## Parameters

- `BEFORE_PATH`, `AFTER_PATH` — the datasets before and after the filter (S3 folders of parquet files).
- `STORAGE_OPTIONS` — Polars S3 options; leave `None` unless the region isn't found.
- `HISTOGRAM_BINS` — number of equal-width bins in the distribution table.
- `LARGEST_CHANGES` — how many rows to show in the "largest changes" tables.
- `KEY_COLUMNS` — columns that identify a row; the datasets are matched on these. Keys must be unique in
  each dataset, otherwise the join fans out. Must include `DATE_COLUMN` and `FILTER_COLUMN`.
- `DATE_COLUMN` — the time axis for the trendline.
- `FILTER_COLUMN`, `FILTERED_VALUES` — the column the filter acts on and the value(s) being filtered
  (here: a job-role column and the job roles removed).
- `VALUE_COLUMN` — the numeric column the filter nulls; used for nulled rows, trendline and distribution.
- `ESTIMATE_COLUMN` — the numeric final estimates column whose change is measured.

In [ ]:
BEFORE_PATH = "s3://<bucket>/domain=<domain>/dataset=<dataset>/version=<before_version>/"
AFTER_PATH = "s3://<bucket>/domain=<domain>/dataset=<dataset>/version=<after_version>/"
STORAGE_OPTIONS = None  # e.g. {"aws_region": "eu-west-2"} if Polars can't find the region
HISTOGRAM_BINS = 10
LARGEST_CHANGES = 15

# Column names are defined once, here. Edit them for your datasets.
KEY_COLUMNS = ["location_id", "import_date", "job_role"]
DATE_COLUMN = "import_date"
FILTER_COLUMN = "job_role"
FILTERED_VALUES = ["senior_care_worker"]
VALUE_COLUMN = "job_role_count"
ESTIMATE_COLUMN = "estimate_filled_posts"
VALUE_BEFORE, VALUE_AFTER = f"{VALUE_COLUMN}_before", f"{VALUE_COLUMN}_after"
ESTIMATE_BEFORE, ESTIMATE_AFTER = f"{ESTIMATE_COLUMN}_before", f"{ESTIMATE_COLUMN}_after"


## Setup

In [ ]:
import polars as pl

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(20)

## Match the datasets

Scans both datasets lazily, keeps only the columns used, and full-joins them on `KEY_COLUMNS`. Value
columns get a `_before` or `_after` suffix, and `filtered` marks rows whose `FILTER_COLUMN` is in
`FILTERED_VALUES`. The coverage table counts rows in both datasets, only before (dropped by the filter)
and only after (new rows, unexpected for a filter).

In [ ]:
def scan(path: str, suffix: str) -> pl.LazyFrame:
    """Scan a dataset, keeping the used columns and suffixing the value columns."""
    columns = list(dict.fromkeys(KEY_COLUMNS + [VALUE_COLUMN, ESTIMATE_COLUMN]))
    lf = pl.scan_parquet(path, storage_options=STORAGE_OPTIONS).select(columns)
    return lf.rename({c: f"{c}_{suffix}" for c in (VALUE_COLUMN, ESTIMATE_COLUMN)}).with_columns(
        pl.lit(True).alias(f"in_{suffix}")
    )


joined_lf = (
    scan(BEFORE_PATH, "before")
    .join(scan(AFTER_PATH, "after"), on=KEY_COLUMNS, how="full", coalesce=True, nulls_equal=True)
    .with_columns(
        pl.col(FILTER_COLUMN).is_in(FILTERED_VALUES).alias("filtered"),
        pl.col("in_before", "in_after").fill_null(False),
    )
)

joined_lf.group_by("in_before", "in_after").agg(pl.len().alias("rows")).sort("in_before", "in_after").collect()

## 1. Rows nulled by the filter

A row is **nulled** when `VALUE_COLUMN` was present before and is null after. The first table covers rows
in the filtered values and all other rows: rows with a value before, rows nulled and the share nulled.
Expect most filtered rows nulled and **no other rows nulled**; a non-zero count for the other rows means
the filter reached beyond its target. The second table lists the `FILTER_COLUMN` values that lost the most rows.

In [ ]:
nulled = pl.col(VALUE_BEFORE).is_not_null() & pl.col(VALUE_AFTER).is_null()

nulled_summary_df = (
    joined_lf.group_by("filtered")
    .agg(
        pl.col(VALUE_BEFORE).count().alias("rows_with_value_before"),
        nulled.sum().alias("rows_nulled"),
    )
    .with_columns((pl.col("rows_nulled") / pl.col("rows_with_value_before")).alias("share_nulled"))
    .sort("filtered", descending=True)
    .collect()
)
print(nulled_summary_df)

(
    joined_lf.group_by(FILTER_COLUMN, "filtered")
    .agg(nulled.sum().alias("rows_nulled"))
    .filter(pl.col("rows_nulled") > 0)
    .sort("rows_nulled", descending=True)
    .head(LARGEST_CHANGES)
    .collect()
)

## 2. Trendline over time

Mean of `VALUE_COLUMN` per `DATE_COLUMN`, before and after, for all rows and for the filtered rows only.
`difference` is after minus before. A trendline that moves only in the filtered rows is the intended
effect; a shift in the all-rows lines is the filter's effect on the overall level. A `null` mean means no
values are left on that date.

In [ ]:
def trendline(rows: pl.Expr, label: str) -> pl.LazyFrame:
    """Mean of the value column per date, before and after, over the selected rows."""
    return (
        joined_lf.filter(rows)
        .group_by(DATE_COLUMN)
        .agg(
            pl.col(VALUE_BEFORE).mean().alias(f"{label}_before"),
            pl.col(VALUE_AFTER).mean().alias(f"{label}_after"),
        )
        .with_columns((pl.col(f"{label}_after") - pl.col(f"{label}_before")).alias(f"{label}_difference"))
    )


trend_df = (
    trendline(pl.lit(True), "all")
    .join(trendline(pl.col("filtered"), "filtered"), on=DATE_COLUMN, how="left", nulls_equal=True)
    .sort(DATE_COLUMN)
    .collect()
)
trend_df

## 3. Value distributions

Summary statistics of `VALUE_COLUMN` before and after, then a histogram of `HISTOGRAM_BINS` equal-width
bins over the combined range (`bin_from` is each bin's lower edge). `count_difference` is after minus
before: negative bins show where the filter removed values. Null values are counted in `nulls` but not in
the histogram.

In [ ]:
value_stats_df = pl.concat([
    joined_lf.select(
        pl.lit(label).alias("dataset"),
        pl.col(column).null_count().alias("nulls"),
        pl.col(column).count().alias("values"),
        pl.col(column).mean().alias("mean"),
        pl.col(column).std().alias("std"),
        pl.col(column).min().alias("min"),
        *[pl.col(column).quantile(q).alias(f"p{int(q * 100)}") for q in (0.25, 0.5, 0.75)],
        pl.col(column).max().alias("max"),
    )
    for label, column in (("before", VALUE_BEFORE), ("after", VALUE_AFTER))
]).collect()
value_stats_df

Histogram: rows per bin before and after.

In [ ]:
low, high = (
    joined_lf.select(
        pl.min_horizontal(pl.col(VALUE_BEFORE).min(), pl.col(VALUE_AFTER).min()).alias("low"),
        pl.max_horizontal(pl.col(VALUE_BEFORE).max(), pl.col(VALUE_AFTER).max()).alias("high"),
    )
    .collect()
    .row(0)
)
width = (high - low) / HISTOGRAM_BINS or 1  # one repeated value gets bins of width 1


def bin_counts(column: str, label: str) -> pl.LazyFrame:
    """Rows per histogram bin for one value column."""
    bin_index = ((pl.col(column) - low) / width).floor().clip(0, HISTOGRAM_BINS - 1).cast(pl.Int32)
    return (
        joined_lf.filter(pl.col(column).is_not_null())
        .group_by(bin_index.alias("bin"))
        .agg(pl.len().alias(f"count_{label}"))
    )


histogram_df = (
    pl.LazyFrame({"bin": range(HISTOGRAM_BINS)}, schema={"bin": pl.Int32})
    .join(bin_counts(VALUE_BEFORE, "before"), on="bin", how="left")
    .join(bin_counts(VALUE_AFTER, "after"), on="bin", how="left")
    .with_columns(
        (low + pl.col("bin") * width).alias("bin_from"),
        pl.col("count_before", "count_after").fill_null(0).cast(pl.Int64),
    )
    .with_columns((pl.col("count_after") - pl.col("count_before")).alias("count_difference"))
    .select("bin", "bin_from", "count_before", "count_after", "count_difference")
    .sort("bin")
    .collect()
)
histogram_df

## 4. Effect on the final estimates

Compares `ESTIMATE_COLUMN` before and after, for filtered rows and all other rows. `rows_changed` counts
rows whose estimate differs (a null differs from a value); `total_difference` is the after total minus the
before total and `relative_difference` is that as a share of the before total. Expect the change in the
filtered rows; a change in the other rows is a knock-on effect, for example through an aggregate that used
the filtered values. The second table shows the `FILTER_COLUMN` values whose totals moved most.

In [ ]:
def estimate_effect(*by: str) -> pl.LazyFrame:
    """Rows changed and estimate totals before and after, grouped by `by`."""
    difference = pl.col("total_after") - pl.col("total_before")
    return (
        joined_lf.group_by(*by)
        .agg(
            pl.col(ESTIMATE_BEFORE).ne_missing(pl.col(ESTIMATE_AFTER)).sum().alias("rows_changed"),
            pl.col(ESTIMATE_BEFORE).sum().alias("total_before"),
            pl.col(ESTIMATE_AFTER).sum().alias("total_after"),
        )
        .with_columns(
            difference.alias("total_difference"),
            (difference / pl.col("total_before")).alias("relative_difference"),
        )
    )


estimate_summary_df = estimate_effect("filtered").sort("filtered", descending=True).collect()
print(estimate_summary_df)

(
    estimate_effect(FILTER_COLUMN, "filtered")
    .filter(pl.col("rows_changed") > 0)
    .sort(pl.col("total_difference").abs(), descending=True)
    .head(LARGEST_CHANGES)
    .collect()
)

## Results

Headline numbers: rows nulled and the change in the estimates total, for filtered and other rows. Rows
nulled, or estimates changed, outside the filtered values deserve a closer look.

In [ ]:
nulled_summary_df.select("filtered", "rows_nulled").join(
    estimate_summary_df.select("filtered", "rows_changed", "total_difference", "relative_difference"),
    on="filtered",
).sort("filtered", descending=True)